<h1 style="color:red; font-size=40px; font-weight: bolder; letter-spacing:10px; text-align:center">ETL</h1>

<h2>Objectives</h2>

- Download required zip file from given url
- Etract all files from zipfile
- Extract data from files into one large dataframe
- Transfrom data
- load transfromed data into a csv file
- log each process in a log file

In [18]:
#logger function
from datetime import datetime
def log_progress(message, log_file = "log_file.txt"): 
    timestamp_format = '%Y-%h-%d-%H:%M:%S' # Year-Monthname-Day-Hour-Minute-Second 
    now = datetime.now() # get current timestamp 
    timestamp = now.strftime(timestamp_format) 
    with open(log_file,"a") as f: 
        f.write(timestamp + ',' + message + '\n') 


<h3>Download and Extract files</h3>

In [24]:
#retrieve source files
import urllib

url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-PY0221EN-SkillsNetwork/labs/module%206/Lab%20-%20Extract%20Transform%20Load/data/source.zip"

filename = "source.zip"
#start download and log progress
log_progress("Starting download of source files")

urllib.request.urlretrieve(url, filename)
log_progress("Download completed successfully")


In [3]:
import zipfile, os

#unzip source files into source folder
output_folder = "source"

#create output folder if it doesn't exist
os.makedirs(output_folder, exist_ok=True)
with zipfile.ZipFile("source.zip","r") as zip_ref:
    zip_ref.extractall(output_folder)

In [ ]:
import glob 
import pandas as pd
# functions to extract different file formats

def extract_csv(file_to_process): 
	dataframe = pd.read_csv(file_to_process) 
	return dataframe


def extract_json(file_to_process): 
    dataframe = pd.read_json(file_to_process,lines=True) 
    return dataframe

def extract_xml(file_to_process)->pd.DataFrame:
    """Extracts data from an XML file and returns a DataFrame.""" 
    return pd.read_xml(file_to_process)
    

In [ ]:
# extract data from all files and concatenate into a single DataFrame
def extract_data(path_to_files)->pd.DataFrame:
    """
    Extracts data from all files in the specified directory and concatenates them into a single DataFrame.""" 
    all_files = glob.glob(path_to_files + "/*") 
    dataframe_list = [] 
    for file in all_files: 
        if file.endswith(".csv"): 
            dataframe_list.append(extract_csv(file)) 
        elif file.endswith(".json"): 
            dataframe_list.append(extract_json(file)) 
        elif file.endswith(".xml"): 
            dataframe_list.append(extract_xml(file)) 
    return pd.concat(dataframe_list, ignore_index=True)

#logging the data extraction process
log_progress("Starting data extraction process")        
extract_data(output_folder)
log_progress("Data extraction process completed successfully")


<h3>Transfrom</h3>

In [26]:

def transform(data): 
    '''Convert inches to meters and round off to two decimals 
    1 inch is 0.0254 meters '''
    data['height'] = round(data.height * 0.0254,2) 
 
    '''Convert pounds to kilograms and round off to two decimals 
    1 pound is 0.45359237 kilograms '''
    data['weight'] = round(data.weight * 0.45359237,2) 
    return data 

#logging transformation progress
log_progress("Starting data transformation")
transformed_data = transform(extract_data(output_folder))
log_progress("Data transformation completed successfully")


<h3>Load</h3>

In [27]:
def load_data(target_file, transformed_data): 
    transformed_data.to_csv(target_file, index=False) 

target_file = "transformed_data.csv"

#logging data loading process
log_progress("Starting data loading process")
load_data(target_file, transformed_data)
log_progress("Data loading process completed successfully")

